# Construcción OBT (One Big Table)

Lee de RAW INT_TRIPS_ENRICHED, unifica esquemas, agrega derivadas y escribe a `ANALYTICS.OBT_TRIPS`.

- Grano: 1 fila = 1 viaje
- Idempotencia: DELETE + INSERT por source_year/source_month
- Procesamiento: mes a mes para no saturar memoria

## Imports y configuración

In [ ]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import snowflake.connector
from tqdm.notebook import tqdm as tqdm_notebook

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_RAW          = os.environ.get("SNOWFLAKE_SCHEMA_RAW", "RAW")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

START_YEAR  = int(os.environ.get("START_YEAR", "2015"))
END_YEAR    = int(os.environ.get("END_YEAR", "2025"))
RUN_ID      = os.environ.get("RUN_ID", "run_001")

VALIDATE_NULLS      = os.environ.get("VALIDATE_NULLS", "true").lower() == "true"
VALIDATE_RANGES     = os.environ.get("VALIDATE_RANGES", "true").lower() == "true"
VALIDATE_TIMESTAMPS = os.environ.get("VALIDATE_TIMESTAMPS", "true").lower() == "true"
VALIDATE_CONSISTENCY = os.environ.get("VALIDATE_CONSISTENCY", "true").lower() == "true"

spark = SparkSession.builder \
    .appName("P3_Construccion_OBT") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.extraJavaOptions", "-XX:MaxDirectMemorySize=4g") \
    .config("spark.executor.extraJavaOptions", "-XX:MaxDirectMemorySize=4g") \
    .getOrCreate()

sf_raw = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_RAW,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

sf_analytics = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

def get_snowflake_connection(schema=SCHEMA_ANALYTICS):
    return snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER,
        password=SNOWFLAKE_PASSWORD, database=SNOWFLAKE_DATABASE,
        schema=schema, warehouse=SNOWFLAKE_WAREHOUSE, role=SNOWFLAKE_ROLE,
    )

print(f"Spark {spark.version} | OBT: {SNOWFLAKE_DATABASE}.{SCHEMA_ANALYTICS}.OBT_TRIPS")
print(f"Rango: {START_YEAR}-{END_YEAR} | RUN_ID: {RUN_ID}")

Spark 3.5.0 | OBT: NYC_TLC.ANALYTICS.OBT_TRIPS
Rango: 2015-2025 | RUN_ID: run_001


## Funciones de construcción OBT

In [ ]:
def read_raw_chunk(year, month):
    """Lee un mes de ambos servicios desde INT. Retorna None si no hay datos."""
    table = "INT_TRIPS_ENRICHED"
    try:
        obt_minimum_columns = ["PICKUP_DATETIME", "DROPOFF_DATETIME", 
                       "PU_LOCATION_ID", "PU_ZONE", "PU_BOROUGH",
                       "DO_LOCATION_ID", "DO_ZONE", "DO_BOROUGH",
                       "SERVICE_TYPE", "VENDOR_ID", "VENDOR_NAME",
                       "RATE_CODE_ID", "RATE_CODE_DESC", "PAYMENT_TYPE",
                       "PAYMENT_TYPE_DESC", "TRIP_TYPE", "PASSENGER_COUNT",
                       "TRIP_DISTANCE", "STORE_AND_FWD_FLAG", "FARE_AMOUNT",
                       "EXTRA", "MTA_TAX", "TIP_AMOUNT", "TOLLS_AMOUNT",
                       "IMPROVEMENT_SURCHARGE", "CONGESTION_SURCHARGE",
                       "AIRPORT_FEE", "TOTAL_AMOUNT", "RUN_ID",
                       "INGESTED_AT_UTC", "SOURCE_PATH", "SOURCE_YEAR", "SOURCE_MONTH"]
        
        df = spark.read.format("net.snowflake.spark.snowflake") \
            .options(**sf_raw) \
            .option("query", f"""
                SELECT * FROM {table}
                WHERE SOURCE_YEAR = {year} AND SOURCE_MONTH = {month}
            """).load()
        df_selected = df.select(obt_minimum_columns)
        df_selected = df_selected.toDF(*[c.lower() for c in df_selected.columns])
        if df_selected.head(1):
            return df_selected
    except Exception:
        pass
    return None

def derive(df):
    """Columnas derivadas para la OBT."""
    # Columnas de tiempo derivadas
    df = df.withColumn("pickup_date", F.to_date("pickup_datetime")) \
           .withColumn("pickup_hour", F.hour("pickup_datetime")) \
           .withColumn("dropoff_date", F.to_date("dropoff_datetime")) \
           .withColumn("dropoff_hour", F.hour("dropoff_datetime")) \
           .withColumn("day_of_week", F.dayofweek("pickup_datetime")) \
           .withColumn("month", F.month("pickup_datetime")) \
           .withColumn("year", F.year("pickup_datetime"))

    # Derivadas de viaje
    df = df.withColumn("trip_duration_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60.0
    )
    df = df.withColumn("avg_speed_mph",
        F.when(
            (F.col("trip_duration_min") > 0) & (F.col("trip_distance") > 0),
            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)
        )
    )
    df = df.withColumn("tip_pct",
        F.when(
            F.col("total_amount") > 0,
            (F.col("tip_amount") / F.col("total_amount")) * 100.0
        )
    )

    # Filtrar segun flags de validacion (aqui si se filtran, no en RAW)
    if VALIDATE_NULLS:
        df = df.filter(
            F.col("pickup_datetime").isNotNull() & 
            F.col("dropoff_datetime").isNotNull() &
            F.col("pu_location_id").isNotNull() &
            F.col("do_location_id").isNotNull() &
            F.col("service_type").isNotNull() &
            F.col("total_amount").isNotNull() &
            F.col("trip_distance").isNotNull()
        )
    if VALIDATE_TIMESTAMPS:
        df = df.filter(F.col("pickup_datetime") <= F.col("dropoff_datetime"))
    if VALIDATE_RANGES:
        df = df.filter((F.col("trip_distance") >= 0) | F.col("trip_distance").isNull())
        df = df.filter((F.col("total_amount") >= 0) | F.col("total_amount").isNull())
        df = df.filter((F.col("passenger_count").between(0, 9)) | F.col("passenger_count").isNull())
        df = df.filter((F.col("trip_duration_min").between(0, 1440)) | F.col("trip_duration_min").isNull())
        df = df.filter((F.col("avg_speed_mph").between(0, 120)) | F.col("avg_speed_mph").isNull())
        df = df.filter((F.col("tip_pct").between(0, 100)) | F.col("tip_pct").isNull())
    if VALIDATE_CONSISTENCY:
        df = df.filter(
            (F.col("fare_amount") <= F.col("total_amount")) | 
            F.col("fare_amount").isNull() | 
            F.col("total_amount").isNull()
        )
        df = df.filter(~((F.col("service_type") == "green") & F.col("trip_type").isNull()))
    return df


def delete_obt_partition(year, month):
    """Idempotencia: borra datos del mes en OBT antes de reinsertar."""
    conn = get_snowflake_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
            DELETE FROM {SCHEMA_ANALYTICS}.OBT_TRIPS
            WHERE source_year = {year} AND source_month = {month}
        """)
        return cursor.rowcount
    except Exception:
        return 0
    finally:
        conn.close()


print("Funciones OBT cargadas.")

Funciones OBT cargadas.


## Construcción OBT mes a mes

Para cada mes: lee yellow+green de INT, agrega derivadas y escribe a ANALYTICS.OBT_TRIPS.

In [4]:
obt_log = []
chunks = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]

pbar = tqdm_notebook(chunks, desc="Construccion OBT", unit="mes")

for year, month in pbar:
    pbar.set_postfix_str(f"{year}-{month:02d}")
    start = time.time()

    # Leer ambos servicios para un mes-año
    chunk = read_raw_chunk(year, month)

    if not chunk:
        obt_log.append({"year": year, "month": month, "status": "missing", "rows": 0})
        continue

    try:
        # Derivadas + filtros
        df_obt = derive(chunk)
        row_count = df_obt.count()

        # Idempotencia
        deleted = delete_obt_partition(year, month)
        if deleted > 0:
            print(f"  IDEMPOTENCIA [{year}-{month:02d}]: borradas {deleted:,} filas previas")

        df_repartitioned = df_obt.repartition(20)
        
        # Escribir a OBT
        df_repartitioned.write.format("net.snowflake.spark.snowflake") \
            .options(**sf_analytics) \
            .option("dbtable", "OBT_TRIPS") \
            .mode("append") \
            .save()

        elapsed = round(time.time() - start, 2)
        obt_log.append({"year": year, "month": month, "status": "ok", "rows": row_count, "time_sec": elapsed})
        pbar.set_postfix_str(f"{year}-{month:02d} | {row_count:,} filas | {elapsed}s")

    except Exception as e:
        obt_log.append({"year": year, "month": month, "status": "failed", "rows": 0, "error": str(e)})
        print(f"  ERROR [{year}-{month:02d}]: {e}")

pbar.close()
ok = sum(1 for r in obt_log if r["status"] == "ok")
total = sum(r["rows"] for r in obt_log)
print(f"\nOBT finalizada: {ok} meses ok | {total:,} filas totales")

Construccion OBT:   0%|          | 0/132 [00:00<?, ?mes/s]


OBT finalizada: 132 meses ok | 866,674,985 filas totales


## Verificación: muestra de OBT_TRIPS

In [5]:
df_sample = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_analytics) \
    .option("query", "SELECT * FROM OBT_TRIPS LIMIT 5") \
    .load()

print(f"Columnas OBT: {len(df_sample.columns)}")
for c in sorted(df_sample.columns):
    print(f"  {c}")

df_sample.show(5, truncate=False, vertical=True)

Columnas OBT: 43
  AIRPORT_FEE
  AVG_SPEED_MPH
  CONGESTION_SURCHARGE
  DAY_OF_WEEK
  DO_BOROUGH
  DO_LOCATION_ID
  DO_ZONE
  DROPOFF_DATE
  DROPOFF_DATETIME
  DROPOFF_HOUR
  EXTRA
  FARE_AMOUNT
  IMPROVEMENT_SURCHARGE
  INGESTED_AT_UTC
  MONTH
  MTA_TAX
  PASSENGER_COUNT
  PAYMENT_TYPE
  PAYMENT_TYPE_DESC
  PICKUP_DATE
  PICKUP_DATETIME
  PICKUP_HOUR
  PU_BOROUGH
  PU_LOCATION_ID
  PU_ZONE
  RATE_CODE_DESC
  RATE_CODE_ID
  RUN_ID
  SERVICE_TYPE
  SOURCE_MONTH
  SOURCE_PATH
  SOURCE_YEAR
  STORE_AND_FWD_FLAG
  TIP_AMOUNT
  TIP_PCT
  TOLLS_AMOUNT
  TOTAL_AMOUNT
  TRIP_DISTANCE
  TRIP_DURATION_MIN
  TRIP_TYPE
  VENDOR_ID
  VENDOR_NAME
  YEAR
-RECORD 0------------------------------------------------------------------------------------------------
 PICKUP_DATETIME       | 2015-01-26 08:06:59                                                             
 DROPOFF_DATETIME      | 2015-01-26 09:05:48                                                             
 PU_LOCATION_ID        | 100      

## Conteos por servicio y año

In [8]:
df_counts = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_analytics) \
    .option("query", """
        SELECT service_type, source_year, COUNT(*) as total_rows
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
        ORDER BY service_type, source_year
    """).load()

df_counts.show(50, truncate=False)

+------------+-----------+----------+
|SERVICE_TYPE|SOURCE_YEAR|TOTAL_ROWS|
+------------+-----------+----------+
|green       |2015       |19202652  |
|green       |2016       |16352455  |
|green       |2017       |11710371  |
|green       |2018       |8876633   |
|green       |2019       |6262384   |
|green       |2020       |1728859   |
|green       |2021       |1066639   |
|green       |2022       |838206    |
|green       |2023       |784821    |
|green       |2024       |658042    |
|green       |2025       |588259    |
|yellow      |2015       |145983958 |
|yellow      |2016       |131073961 |
|yellow      |2017       |113442786 |
|yellow      |2018       |102802559 |
|yellow      |2019       |84418343  |
|yellow      |2020       |24534075  |
|yellow      |2021       |30726810  |
|yellow      |2022       |39386784  |
|yellow      |2023       |37930878  |
|yellow      |2024       |40558815  |
|yellow      |2025       |47746695  |
+------------+-----------+----------+



In [14]:
total_count = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_analytics) \
    .option("query", """
        SELECT COUNT(*) as total_rows
        FROM OBT_TRIPS
    """).load()

print(f"Conteo total de filas en la OBT final: {total_count.first()['TOTAL_ROWS']}")

Conteo total de filas en la OBT final: 866674985
